In [19]:
# ==============================================================================
# SEL 1: INSTALASI & IMPORT LIBRARY
# ==============================================================================
!pip install requests beautifulsoup4 pandas numpy tqdm -q

import requests
import json
import re
import time
import os
import warnings
import pandas as pd
from bs4 import BeautifulSoup
from datetime import datetime
from tqdm.notebook import tqdm

warnings.filterwarnings('ignore')
print("✅ Environment siap. Library berhasil dimuat.")

# ==============================================================================
# SEL 2: KONFIGURASI TAKSONOMI BENCANA & ENDPOINT
# ==============================================================================
BNPB_CKAN_BASE = "https://data.bnpb.go.id/api/3/action"
INARISK_BASE   = "https://inarisk.bnpb.go.id/api"

# Daftar berbagai bencana sesuai dokumentasi API BNPB untuk iterasi multi-hazard
DAFTAR_HAZARD = {
    "banjir": "Flood",
    "gempa": "Earthquake",
    "tsunami": "Tsunami",
    "longsor": "Landslide",
    "gunung_api": "Volcanic Eruption",
    "kekeringan": "Drought",
    "kebakaran_hutan": "Forest Fire"
}

# Koordinat pusat provinsi untuk fallback pencarian risiko spasial secara dinamis
FALLBACK_GEO = {
    "aceh": {"lat": 4.1755, "lon": 96.8115},
    "sumatera barat": {"lat": -0.7399, "lon": 100.8000},
    "jawa tengah": {"lat": -7.1509, "lon": 110.1403},
    "jawa timur": {"lat": -7.5360, "lon": 112.2331},
    "kalimantan selatan": {"lat": -3.0926, "lon": 115.2838},
    "dki jakarta": {"lat": -6.2088, "lon": 106.8456},
    "sulawesi tengah": {"lat": -1.4300, "lon": 121.4456}
}

print(f"✅ Taksonomi dimuat untuk {len(DAFTAR_HAZARD)} jenis bencana.")

# ==============================================================================
# SEL 3: TEXT PREPROCESSING & DYNAMIC API INTEGRATION ENGINE
# ==============================================================================
def bersihkan_teks_bnpb(text):
    """Pembersihan teks mentah dari sisa tag HTML/skrip agar bersih saat dibaca LLM"""
    if not text: return "Tidak ada deskripsi teks."
    soup = BeautifulSoup(text, "lxml")
    cleaned = re.sub(r'\s+', ' ', soup.get_text()).strip()
    return cleaned

def deteksi_lokasi_dari_teks(teks):
    """Heuristic Text Extraction untuk menemukan entitas wilayah di dalam teks narasi"""
    teks_lower = teks.lower()
    for prov, coords in FALLBACK_GEO.items():
        if prov in teks_lower:
            return prov, coords["lat"], coords["lon"]
    return "indonesia", -6.2088, 106.8456 # Default Jakarta jika teks bersifat nasional

def dapatkan_risks_inarisk(lat, lon):
    """Tarik data matriks risiko spasial (Tabular API) pendukung teks"""
    try:
        r = requests.get(f"{INARISK_BASE}/risk/score", params={"lat": lat, "lon": lon}, timeout=10)
        if r.status_code == 200:
            return r.json()
    except:
        pass
    return None

print("✅ Preprocessor dan Dynamic Finder siap.")

# ==============================================================================
# SEL 4: PIPELINE ANALISIS TEKS MULTI-BENCANA (LLM CONTEXT GENERATOR)
# ==============================================================================
def running_text_analysis_pipeline():
    all_hazard_corpus = []

    print("⏳ Menarik dokumen dari berbagai silo bencana di CKAN BNPB...")
    for keyword, english_name in DAFTAR_HAZARD.items():
        print(f"🔍 Memproses Teks Bencana: [{keyword.upper()}]")

        # Ekstrak data teks historis dari portal CKAN
        try:
            resp = requests.get(f"{BNPB_CKAN_BASE}/package_search", params={"q": keyword, "rows": 3}, timeout=15)
            if resp.status_code != 200: continue
            datasets = resp.json().get("result", {}).get("results", [])
        except Exception as e:
            print(f"   ⚠ Gagal mengambil data untuk {keyword}: {e}")
            continue

        for doc in datasets:
            judul = doc.get("title", "")
            deskripsi_mentah = doc.get("notes", "")
            deskripsi_bersih = bersihkan_teks_bnpb(deskripsi_mentah)

            # Langkah Kunci: Temukan lokasi dari teks narasi terlebih dahulu
            nama_wilayah, lat, lon = deteksi_lokasi_dari_teks(deskripsi_bersih + " " + judul)

            # Tarik indeks risiko InaRisk berdasarkan lokasi teks yang terdeteksi (Dynamic Cross-Reference)
            data_spasial = dapatkan_risks_inarisk(lat, lon)
            matriks_risiko = data_spasial.get("risks", {}) if data_spasial else {}
            kab_bnpb = data_spasial.get("kabupaten", "Unknown") if data_spasial else "Unknown"

            # Bungkus dokumen ke dalam skema Semantic Layer siap umpan ke LLM (Claude/GPT)
            prompt_payload = (
                f"[SYSTEM INJECTION: DISASTER SEMANTIC EVIDENCE LAYER]\n"
                f"Kategori Utama Bencana: {english_name} ({keyword})\n"
                f"Judul Dokumen BNPB    : {judul}\n"
                f"Narasi Tekstual       : {deskripsi_bersih}\n\n"
                f"--- REFERENSI SILO DATA TABULAR/SPASIAL INARISK ---\n"
                f"Deteksi Lokasi Konteks: {kab_bnpb} (Query Base: Prov. {nama_wilayah.title()})\n"
                f"Matriks Risiko Daerah : {json.dumps(matriks_risiko, ensure_ascii=False)}\n\n"
                f"--- INSTRUKSI AUDIT LLM ---\n"
                f"1. Analisis teks narasi di atas dan ekstrak entitas: Kapan terjadi, objek terdampak, dan aktor terlibat.\n"
                f"2. Hitung 'Semantic Severity Score' (1-5) berdasarkan kedalaman teks narasi.\n"
                f"3. Validasi silang: Apakah isi teks narasi sinkron dengan tingkat risiko '{keyword}' yang tercatat di InaRisk pada daerah tersebut?\n"
                f"Kembalikan output analisis dalam struktur format JSON bersih."
            )

            node_korpus = {
                "bencana_id": doc.get("id"),
                "jenis_hazard": keyword,
                "judul_laporan": judul,
                "teks_narasi_kondisi": deskripsi_bersih,
                "silo_spasial_terkait": {
                    "kabupaten_terdeteksi": kab_bnpb,
                    "koordinat_evaluasi": {"lat": lat, "lon": lon},
                    "skor_risiko_tabular": matriks_risiko.get(keyword, "Tidak Terbaca")
                },
                "llm_ready_prompt": prompt_payload
            }
            all_hazard_corpus.append(node_korpus)
            time.sleep(0.5) # Jeda aman anti rate-limiting

    return all_hazard_corpus

# ==============================================================================
# SEL 5: EKSEKUSI PIPELINE & SHOW OUTPUT ANALISIS
# ==============================================================================
korpus_bencana_final = running_text_analysis_pipeline()

print("\n" + "="*80)
print("📝 HASIL COMPILING KORPUS TEKS MULTI-HAZARD SELESAI")
print("="*80)
print(f"Total Dokumen Berbagai Bencana Berhasil Diproses: {len(korpus_bencana_final)}\n")

# Menampilkan 2 contoh teratas dari kategori bencana yang berbeda untuk inspeksi siap-LLM
for i, item in enumerate(korpus_bencana_final[:2], 1):
    print(f"🔹 DOKUMEN ANALISIS TEKS {i} [{item['jenis_hazard'].upper()}]")
    print(f"   Judul: {item['judul_laporan']}")
    print(f"   Kabupaten Sinkronisasi: {item['silo_spasial_terkait']['kabupaten_terdeteksi']}")
    print(f"   Skor Risiko Tabular   : {item['silo_spasial_terkait']['skor_risiko_tabular']}")
    print(f"   Cuplikan Prompt Siap LLM:\n")
    print(item['llm_ready_prompt'][:800] + "\n... [Teks dipotong untuk efisiensi layar Colab] ...\n")
    print("-" * 80)

# Simpan ke file JSON lokal di Colab
with open("korpus_analisis_teks_bencana.json", "w", encoding="utf-8") as f:
    json.dump(korpus_bencana_final, f, indent=4, ensure_ascii=False)
print("💾 File 'korpus_analisis_teks_bencana.json' berhasil dibuat di storage Google Colab Anda!")

✅ Environment siap. Library berhasil dimuat.
✅ Taksonomi dimuat untuk 7 jenis bencana.
✅ Preprocessor dan Dynamic Finder siap.
⏳ Menarik dokumen dari berbagai silo bencana di CKAN BNPB...
🔍 Memproses Teks Bencana: [BANJIR]
🔍 Memproses Teks Bencana: [GEMPA]
🔍 Memproses Teks Bencana: [TSUNAMI]
🔍 Memproses Teks Bencana: [LONGSOR]
🔍 Memproses Teks Bencana: [GUNUNG_API]
🔍 Memproses Teks Bencana: [KEKERINGAN]
🔍 Memproses Teks Bencana: [KEBAKARAN_HUTAN]

📝 HASIL COMPILING KORPUS TEKS MULTI-HAZARD SELESAI
Total Dokumen Berbagai Bencana Berhasil Diproses: 21

🔹 DOKUMEN ANALISIS TEKS 1 [BANJIR]
   Judul: BANJIR_AR
   Kabupaten Sinkronisasi: Unknown
   Skor Risiko Tabular   : Tidak Terbaca
   Cuplikan Prompt Siap LLM:

[SYSTEM INJECTION: DISASTER SEMANTIC EVIDENCE LAYER]
Kategori Utama Bencana: Flood (banjir)
Judul Dokumen BNPB    : BANJIR_AR
Narasi Tekstual       : Berisi informasi sebaran lokasi banjir dalam bentuk polygon (AR) di Kabupaten Kotabaru Provinsi Kalimantan Selatan. Data ini bersumb

In [20]:
# ==============================================================================
# SEL BARU: TEXT ENTITY EXTRACTION & GEOLOCATION FIXER
# ==============================================================================
import json
import pandas as pd

# Load korpus teks yang berhasil dibuat tadi
with open("korpus_analisis_teks_bencana.json", "r", encoding="utf-8") as f:
    korpus_mentah = json.load(f)

# Kamus koordinat riil pusat kabupaten untuk mengoreksi data 'Unknown'
# Ini bertindak sebagai jembatan spatial anchor sebelum citra satelit memotong (clipping) area
GEOTAG_REGISTRY = {
    "kotabaru": {"lat": -3.2452, "lon": 116.2148, "kab": "KAB. KOTABARU", "prov": "KALIMANTAN SELATAN"},
    "demak": {"lat": -6.8914, "lon": 110.6397, "kab": "KAB. DEMAK", "prov": "JAWA TENGAH"},
    "kudus": {"lat": -6.8048, "lon": 110.8407, "kab": "KAB. KUDUS", "prov": "JAWA TENGAH"},
    "jakarta": {"lat": -6.2088, "lon": 106.8456, "kab": "KOTA JAKARTA SELATAN", "prov": "DKI JAKARTA"}
}

hasil_ekstraksi_teks = []

print("🧠 Memulai Proses Ekstraksi Entitas Teks (Text Semantic Extraction)...")

for item in korpus_mentah:
    teks = (item["judul_laporan"] + " " + item["teks_narasi_kondisi"]).lower()

    # 1. Default Extraction Value
    tahun_kejadian = 2024  # Fallback tahun default jika tidak terdeteksi
    kabupaten_fix = "NASIONAL (MULTI_WILAYAH)"
    provinsi_fix = "INDONESIA"
    latitude_fix = -6.2088
    longitude_fix = 106.8456

    # 2. RegEx Rule-Based Extraction untuk mendeteksi Tahun di dalam teks
    match_tahun = re.search(r'\b(20\d{2})\b', teks)
    if match_tahun:
        tahun_kejadian = int(match_tahun.group(1))

    # 3. Geo-Parsing & Koordinat Alignment
    for keyword, geo in GEOTAG_REGISTRY.items():
        if keyword in teks:
            kabupaten_fix = geo["kab"]
            provinsi_fix = geo["prov"]
            latitude_fix = geo["lat"]
            longitude_fix = geo["lon"]
            break

    # 4. Semantic Severity Scoring (Aturan berbasis bobot kata dampak dalam narasi)
    severity_score = 1
    if "rawan" in teks or "sebaran" in teks: severity_score = 2
    if "relawan" in teks or "posko" in teks: severity_score = 3
    if "rusak" in teks or "korban" in teks: severity_score = 4
    if "lumpuh" in teks or "meninggal" in teks: severity_score = 5

    # Susun ke dalam entitas data spasio-semantik bersih
    hasil_ekstraksi_teks.append({
        "bencana_id": item["bencana_id"],
        "jenis_hazard": item["jenis_hazard"].upper(),
        "tahun_kejadian": tahun_kejadian,
        "provinsi": provinsi_fix,
        "kabupaten_kota": kabupaten_fix,
        "latitude": latitude_fix,
        "longitude": longitude_fix,
        "narasi_clean": item["teks_narasi_kondisi"],
        "text_severity_score": severity_score
    })

# Konversi ke Pandas DataFrame agar siap ditarungkan dengan Excel DIBI
df_teks = pd.DataFrame(hasil_ekstraksi_teks)
print("✅ Ekstraksi Teks selesai! Menampilkan 3 baris teratas hasil parsing teks terstruktur:")
display(df_teks.head(3))

# Simpan untuk checkpoint integrasi tri-silo
df_teks.to_csv("checkpoint_silo_teks.csv", index=False)

🧠 Memulai Proses Ekstraksi Entitas Teks (Text Semantic Extraction)...
✅ Ekstraksi Teks selesai! Menampilkan 3 baris teratas hasil parsing teks terstruktur:


,bencana_id,jenis_hazard,tahun_kejadian,provinsi,kabupaten_kota,latitude,longitude,narasi_clean,text_severity_score
0,40ff6569-0115-4d62-b866-47f492ade464,BANJIR,2022,KALIMANTAN SELATAN,KAB. KOTABARU,-3.2452,116.2148,Berisi informasi sebaran lokasi banjir dalam b...,2
1,fea255c2-cef3-46ef-9016-1e798cefdb4f,BANJIR,2024,INDONESIA,NASIONAL (MULTI_WILAYAH),-6.2088,106.8456,Jumlah Titik Daerah Banjir yang terdata,2
2,a53e6e6e-b376-4714-98aa-aaf3a9779121,BANJIR,2024,JAWA TENGAH,KAB. DEMAK,-6.8914,110.6397,Platform Desk Relawan Banjir Longsor Demak-Kud...,3


In [24]:
# ==============================================================================
# SEL PERBAIKAN: INTEGRASI ADAPTIF (TEKS BNPB + TREN TAHUNAN DIBI ASLI)
# ==============================================================================
import pandas as pd
import json
import os

FILE_DIBI = "/content/DIBI.xlsx"

if not os.path.exists(FILE_DIBI):
    print(f"❌ File {FILE_DIBI} tidak ditemukan! Pastikan sudah terunggah di folder /content/")
else:
    # 1. Muat Data Excel DIBI Asli
    df_dibi = pd.read_excel(FILE_DIBI)
    print("✅ File DIBI.xlsx Berhasil Dimuat.")

    # 2. Muat Checkpoint Hasil Ekstraksi Teks BNPB
    df_teks = pd.read_csv("checkpoint_silo_teks.csv")

    # Mapping nama hazard dari teks BNPB ke nama kolom di Excel DIBI Anda
    HAZARD_MAPPING = {
        "BANJIR": "Banjir",
        "GEMPA": "Gempabumi",
        "TSUNAMI": "Tsunami",
        "LONGSOR": "Longsor",
        "GUNUNG_API": "Erupsi gunung api",
        "KEKERINGAN": "Kekeringan",
        "KEBAKARAN_HUTAN": "Kebakaran hutan dan lahan"
    }

    # 3. Proses Penggabungan (Looping Baris Teks untuk Menarik Statistik DIBI)
    print("🔄 Menyinkronkan narasi teks dengan statistik tren tahunan DIBI...")

    total_kejadian_dibi = []

    for idx, row in df_teks.iterrows():
        tahun_target = int(row["tahun_kejadian"])
        hazard_target = row["jenis_hazard"]

        # Cari kolom DIBI yang sesuai dengan jenis hazard
        kolom_dibi_target = HAZARD_MAPPING.get(hazard_target, None)

        # Filter baris di Excel DIBI yang tahunnya cocok
        match_dibi = df_dibi[df_dibi["tahun"] == tahun_target]

        if not match_dibi.empty and kolom_dibi_target in df_dibi.columns:
            # Ambil nilai jumlah kejadian bencana dari excel DIBI
            angka_kejadian = match_dibi[kolom_dibi_target].values[0]
            total_kejadian_dibi.append(angka_kejadian)
        else:
            # Jika tahun atau kolom tidak ketemu di excel DIBI
            total_kejadian_dibi.append(None)

    # Masukkan hasil pencocokan kembali ke DataFrame utama
    df_teks["dibi_total_kejadian_nasional"] = total_kejadian_dibi

    # 4. Formulasi Status Validasi (Cross-Check)
    def buat_status_validasi(row):
        if pd.isna(row["dibi_total_kejadian_nasional"]):
            return "⚠️ DATA TAHUNAN DIBI TIDAK TERSEDIA"
        if row["dibi_total_kejadian_nasional"] == 0:
            return "🚨 ANOMALI: Ada dokumen teks, tapi rekam kejadian di DIBI bernilai 0!"
        return "✅ SINKRON: Terpeta di dalam tren historis DIBI"

    df_teks["status_sinkronisasi_dibi"] = df_teks.apply(buat_status_validasi, axis=1)

    # Tampilkan hasil akhir integrasi 2 silo
    print("\n" + "="*90)
    print("🔗 MATRIKS SINKRONISASI 2-SILO (TEKS PORTAL BNPB + TREN TAHUNAN EXCEL DIBI)")
    print("="*90)
    display(df_teks[["jenis_hazard", "tahun_kejadian", "kabupaten_kota", "text_severity_score", "dibi_total_kejadian_nasional", "status_sinkronisasi_dibi"]].head(10))

    # Simpan hasil final
    df_teks.to_csv("master_teks_dan_dibi_tren.csv", index=False)
    print("\n💾 Checkpoint aman! Hasil integrasi disimpan di 'master_teks_dan_dibi_tren.csv'")

✅ File DIBI.xlsx Berhasil Dimuat.
🔄 Menyinkronkan narasi teks dengan statistik tren tahunan DIBI...

🔗 MATRIKS SINKRONISASI 2-SILO (TEKS PORTAL BNPB + TREN TAHUNAN EXCEL DIBI)


,jenis_hazard,tahun_kejadian,kabupaten_kota,text_severity_score,dibi_total_kejadian_nasional,status_sinkronisasi_dibi
0,BANJIR,2022,KAB. KOTABARU,2,1532.0,✅ SINKRON: Terpeta di dalam tren historis DIBI
1,BANJIR,2024,NASIONAL (MULTI_WILAYAH),2,1457.0,✅ SINKRON: Terpeta di dalam tren historis DIBI
2,BANJIR,2024,KAB. DEMAK,3,1457.0,✅ SINKRON: Terpeta di dalam tren historis DIBI
3,GEMPA,2023,NASIONAL (MULTI_WILAYAH),1,31.0,✅ SINKRON: Terpeta di dalam tren historis DIBI
4,GEMPA,2017,NASIONAL (MULTI_WILAYAH),2,20.0,✅ SINKRON: Terpeta di dalam tren historis DIBI
5,GEMPA,2024,NASIONAL (MULTI_WILAYAH),2,18.0,✅ SINKRON: Terpeta di dalam tren historis DIBI
6,TSUNAMI,2024,NASIONAL (MULTI_WILAYAH),2,NaN,⚠️ DATA TAHUNAN DIBI TIDAK TERSEDIA
7,TSUNAMI,2024,NASIONAL (MULTI_WILAYAH),2,NaN,⚠️ DATA TAHUNAN DIBI TIDAK TERSEDIA
8,TSUNAMI,2021,NASIONAL (MULTI_WILAYAH),1,NaN,⚠️ DATA TAHUNAN DIBI TIDAK TERSEDIA
9,LONGSOR,2017,NASIONAL (MULTI_WILAYAH),1,848.0,✅ SINKRON: Terpeta di dalam tren historis DIBI



💾 Checkpoint aman! Hasil integrasi disimpan di 'master_teks_dan_dibi_tren.csv'


In [28]:
import pandas as pd

# Sesuaikan nama file kalau di Colab namanya berbeda (misal: BMKG.csv atau data_bmkg.csv)
FILE_BMKG_CSV = "/content/BMKG.csv"

try:
    # Membaca file CSV BMKG
    df_bmkg = pd.read_csv(FILE_BMKG_CSV)
    print("✅ File CSV BMKG berhasil dibaca dengan mulus!")
    print("📌 Kolom yang terdeteksi di data BMKG Anda:")
    print(list(df_bmkg.columns))
    print("\n📌 Mengintip 3 baris data teratas:")
    display(df_bmkg.head(3))
except Exception as e:
    print(f"⚠️ Gagal membaca file CSV BMKG. Coba cek lagi nama filenya di menu kiri Colab. Error: {e}")

✅ File CSV BMKG berhasil dibaca dengan mulus!
📌 Kolom yang terdeteksi di data BMKG Anda:
['provinsi', 'tahun', 'bulan', 'suhu_rata_c', 'suhu_max_c', 'suhu_min_c', 'curah_hujan_mm', 'kelembapan_pct', 'kec_angin_ms', 'sumber', 'nama_bulan']

📌 Mengintip 3 baris data teratas:


,provinsi,tahun,bulan,suhu_rata_c,suhu_max_c,suhu_min_c,curah_hujan_mm,kelembapan_pct,kec_angin_ms,sumber,nama_bulan
0,Aceh,2015,1,26.01,28.84,24.00,202.6,85.77,3.62,Open-Meteo ERA5,Januari
1,Aceh,2015,2,26.36,29.42,24.04,48.5,80.93,4.75,Open-Meteo ERA5,Februari
2,Aceh,2015,3,26.95,30.19,24.36,139.0,83.81,3.17,Open-Meteo ERA5,Maret


In [29]:
# ==============================================================================
# SEL MASTER: INTEGRASI MULTI-SILO FINAL (TEKS + DIBI + BMKG CSV)
# ==============================================================================
import pandas as pd
import os

FILE_BMKG_CSV = "/content/BMKG.csv"
FILE_MASTER_2SILO = "master_teks_dan_dibi_tren.csv"

if not os.path.exists(FILE_BMKG_CSV):
    print(f"❌ File '{FILE_BMKG_CSV}' tidak ditemukan di folder Colab. Pastikan nama filenya sudah pas ya!")
else:
    # 1. Muat Data BMKG dari CSV
    df_bmkg = pd.read_csv(FILE_BMKG_CSV)
    print("✅ File CSV BMKG Berhasil Dimuat.")

    print("\n📌 Struktur kolom yang terdeteksi di data BMKG Anda:")
    print(list(df_bmkg.columns))
    print("-" * 60)

    # 2. Muat Base Data Master yang sudah gabung DIBI kemarin
    df_master = pd.read_csv(FILE_MASTER_2SILO)

    # 3. OTOMASI PENCOCOKAN DENGAN BMKG
    # Kita asumsikan CSV BMKG memiliki kolom 'tahun' atau 'TAHUN' untuk pengunci makro
    # Cari nama kolom tahun di BMKG secara adaptif (case-insensitive)
    kolom_tahun_bmkg = None
    for col in df_bmkg.columns:
        if col.lower() == 'tahun':
            kolom_tahun_bmkg = col
            break

    if kolom_tahun_bmkg:
        print(f"🔄 Sinkronisasi Tri-Silo menggunakan pengunci kolom waktu: [{kolom_tahun_bmkg}]")

        # Skenario A: Jika BMKG juga merupakan tren parameter tahunan
        # Kita gabungkan data BMKG ke tabel master berdasarkan kesamaan Tahun
        df_final_tri_silo = pd.merge(
            df_master,
            df_bmkg,
            left_on="tahun_kejadian",
            right_on=kolom_tahun_bmkg,
            how="left"
        )

        # Bersihkan kolom duplikat pasca-merge jika ada
        if kolom_tahun_bmkg != "tahun_kejadian" and kolom_tahun_bmkg in df_final_tri_silo.columns:
            df_final_tri_silo.drop(columns=[kolom_tahun_bmkg], inplace=True)

    else:
        print("⚠️ Kolom berbasis 'tahun' tidak ditemukan di BMKG. Menggabungkan secara urutan baris sebagai fallback.")
        # Skenario B: Jika tidak ada kolom tahun, kita tempelkan kolom BMKG langsung di sampingnya
        df_final_tri_silo = pd.concat([df_master, df_bmkg.iloc[:len(df_master)].reset_index(drop=True)], axis=1)

    # 4. TAMPILKAN MATRIKS DATA INTEGRASI TOTAL
    print("\n" + "="*95)
    print("🚀 BINGO! MATRIKS TRIANGULASI 3-SILO DATA (TEKS BNPB + TABULAR DIBI + PARAMETER BMKG)")
    print("="*95)

    # Memaksa pandas menampilkan semua baris ke bawah tanpa dipotong Colab!
    pd.set_option('display.max_rows', 50)
    display(df_final_tri_silo)

    # 5. SIMPAN DATABASE MASTER UTAMA
    df_final_tri_silo.to_csv("DATABASE_MASTER_TRI_SILO.csv", index=False)
    print("\n💾 SUPER CHECKPOINT! 'DATABASE_MASTER_TRI_SILO.csv' berhasil dikunci dan siap diunduh!")

✅ File CSV BMKG Berhasil Dimuat.

📌 Struktur kolom yang terdeteksi di data BMKG Anda:
['provinsi', 'tahun', 'bulan', 'suhu_rata_c', 'suhu_max_c', 'suhu_min_c', 'curah_hujan_mm', 'kelembapan_pct', 'kec_angin_ms', 'sumber', 'nama_bulan']
------------------------------------------------------------
🔄 Sinkronisasi Tri-Silo menggunakan pengunci kolom waktu: [tahun]

🚀 BINGO! MATRIKS TRIANGULASI 3-SILO DATA (TEKS BNPB + TABULAR DIBI + PARAMETER BMKG)


,bencana_id,jenis_hazard,tahun_kejadian,provinsi_x,kabupaten_kota,latitude,longitude,narasi_clean,text_severity_score,dibi_total_kejadian_nasional,...,provinsi_y,bulan,suhu_rata_c,suhu_max_c,suhu_min_c,curah_hujan_mm,kelembapan_pct,kec_angin_ms,sumber,nama_bulan
0,40ff6569-0115-4d62-b866-47f492ade464,BANJIR,2022,KALIMANTAN SELATAN,KAB. KOTABARU,-3.2452,116.2148,Berisi informasi sebaran lokasi banjir dalam b...,2,1532.0,...,Aceh,1,25.55,29.06,22.86,124.8,82.35,3.91,Open-Meteo ERA5,Januari
1,40ff6569-0115-4d62-b866-47f492ade464,BANJIR,2022,KALIMANTAN SELATAN,KAB. KOTABARU,-3.2452,116.2148,Berisi informasi sebaran lokasi banjir dalam b...,2,1532.0,...,Aceh,2,25.51,29.03,23.11,217.3,84.71,3.87,Open-Meteo ERA5,Februari
2,40ff6569-0115-4d62-b866-47f492ade464,BANJIR,2022,KALIMANTAN SELATAN,KAB. KOTABARU,-3.2452,116.2148,Berisi informasi sebaran lokasi banjir dalam b...,2,1532.0,...,Aceh,3,25.84,29.12,23.31,214.0,85.65,3.19,Open-Meteo ERA5,Maret
3,40ff6569-0115-4d62-b866-47f492ade464,BANJIR,2022,KALIMANTAN SELATAN,KAB. KOTABARU,-3.2452,116.2148,Berisi informasi sebaran lokasi banjir dalam b...,2,1532.0,...,Aceh,4,26.42,29.64,23.86,134.0,83.97,3.09,Open-Meteo ERA5,April
4,40ff6569-0115-4d62-b866-47f492ade464,BANJIR,2022,KALIMANTAN SELATAN,KAB. KOTABARU,-3.2452,116.2148,Berisi informasi sebaran lokasi banjir dalam b...,2,1532.0,...,Aceh,5,27.23,30.14,24.86,124.6,79.19,4.92,Open-Meteo ERA5,Mei
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9571,6d0095c2-8607-4a64-bae8-30518f44800b,KEBAKARAN_HUTAN,2023,INDONESIA,NASIONAL (MULTI_WILAYAH),-6.2088,106.8456,data luas kawasan hutan yang dikelola oleh mas...,1,2051.0,...,Sumatera Utara,8,25.76,29.95,23.07,466.8,90.45,3.05,Open-Meteo ERA5,Agustus
9572,6d0095c2-8607-4a64-bae8-30518f44800b,KEBAKARAN_HUTAN,2023,INDONESIA,NASIONAL (MULTI_WILAYAH),-6.2088,106.8456,data luas kawasan hutan yang dikelola oleh mas...,1,2051.0,...,Sumatera Utara,9,26.13,30.30,23.33,442.4,89.60,3.22,Open-Meteo ERA5,September
9573,6d0095c2-8607-4a64-bae8-30518f44800b,KEBAKARAN_HUTAN,2023,INDONESIA,NASIONAL (MULTI_WILAYAH),-6.2088,106.8456,data luas kawasan hutan yang dikelola oleh mas...,1,2051.0,...,Sumatera Utara,10,25.52,29.42,22.93,602.5,92.16,3.07,Open-Meteo ERA5,Oktober
9574,6d0095c2-8607-4a64-bae8-30518f44800b,KEBAKARAN_HUTAN,2023,INDONESIA,NASIONAL (MULTI_WILAYAH),-6.2088,106.8456,data luas kawasan hutan yang dikelola oleh mas...,1,2051.0,...,Sumatera Utara,11,25.34,29.06,22.97,614.0,92.23,3.13,Open-Meteo ERA5,November



💾 SUPER CHECKPOINT! 'DATABASE_MASTER_TRI_SILO.csv' berhasil dikunci dan siap diunduh!


In [30]:
# ==============================================================================
# SEL: REFINEMENT TRI-SILO (CLEANING SPATIAL ALIGNMENT BMKG)
# ==============================================================================
import pandas as pd

# Load database yang meledak tadi
df_ledak = pd.read_csv("DATABASE_MASTER_TRI_SILO.csv")

print(f"📊 Total baris sebelum dibersihkan: {len(df_ledak)} baris.")

# 1. Standarisasi String Provinsi agar bisa dicocokkan (Huruf besar semua & hapus spasi)
df_ledak['provinsi_x_clean'] = df_ledak['provinsi_x'].astype(str).str.upper().str.strip()
df_ledak['provinsi_y_clean'] = df_ledak['provinsi_y'].astype(str).str.upper().str.strip()

# 2. FILTER 1: COCOKKAN WILAYAH SPASIAL
# Dokumentasi: Hanya loloskan baris jika Provinsi di BNPB cocok dengan Provinsi di BMKG,
# ATAU jika dokumen teksnya bersifat NASIONAL (Maka semua provinsi boleh masuk sebagai tren makro)
df_clean = df_ledak[
    (df_ledak['provinsi_x_clean'] == df_ledak['provinsi_y_clean']) |
    (df_ledak['provinsi_x_clean'] == "INDONESIA")
].copy()

# 3. FILTER 2: TEMPORAL RELEVANCE (Opsional - Fokus ke Bulan Puncak jika data spasial lokal)
# Untuk dokumen lokal seperti KAB. KOTABARU, kita pastikan data iklim yang diambil relevan
# Hapus kolom bantu agar rapi kembali
df_clean.drop(columns=['provinsi_x_clean', 'provinsi_y_clean'], inplace=True)

# Rename kolom provinsi agar tidak membingungkan (x = BNPB asli, y = BMKG Match)
df_clean.rename(columns={'provinsi_x': 'provinsi_bnpb', 'provinsi_y': 'provinsi_climatology'}, inplace=True)

print(f"✅ Pembersihan Selesai! Total baris sekarang: {len(df_clean)} baris.")
print("\n" + "="*95)
print("🎯 DATABASE TRI-SILO FINAL (SUDAH DISARING BERDASARKAN KESESUAIAN WILAYAH)")
print("="*95)

# Tampilkan data hasil saringan
display(df_clean[['jenis_hazard', 'tahun_kejadian', 'provinsi_bnpb', 'kabupaten_kota', 'provinsi_climatology', 'nama_bulan', 'curah_hujan_mm', 'status_sinkronisasi_dibi']].head(15))

# Simpan ke file final yang siap disetor ke modul Citra Satelit / LLM Agent
df_clean.to_csv("DATABASE_FINAL_TRI_SILO_READY.csv", index=False)
print("\n💾 File bersih siap pakai disimpan dengan nama: 'DATABASE_FINAL_TRI_SILO_READY.csv'")

📊 Total baris sebelum dibersihkan: 9576 baris.
✅ Pembersihan Selesai! Total baris sekarang: 8688 baris.

🎯 DATABASE TRI-SILO FINAL (SUDAH DISARING BERDASARKAN KESESUAIAN WILAYAH)


,jenis_hazard,tahun_kejadian,provinsi_bnpb,kabupaten_kota,provinsi_climatology,nama_bulan,curah_hujan_mm,status_sinkronisasi_dibi
144,BANJIR,2022,KALIMANTAN SELATAN,KAB. KOTABARU,Kalimantan Selatan,Januari,346.7,✅ SINKRON: Terpeta di dalam tren historis DIBI
145,BANJIR,2022,KALIMANTAN SELATAN,KAB. KOTABARU,Kalimantan Selatan,Februari,432.5,✅ SINKRON: Terpeta di dalam tren historis DIBI
146,BANJIR,2022,KALIMANTAN SELATAN,KAB. KOTABARU,Kalimantan Selatan,Maret,499.0,✅ SINKRON: Terpeta di dalam tren historis DIBI
147,BANJIR,2022,KALIMANTAN SELATAN,KAB. KOTABARU,Kalimantan Selatan,April,311.2,✅ SINKRON: Terpeta di dalam tren historis DIBI
148,BANJIR,2022,KALIMANTAN SELATAN,KAB. KOTABARU,Kalimantan Selatan,Mei,440.4,✅ SINKRON: Terpeta di dalam tren historis DIBI
149,BANJIR,2022,KALIMANTAN SELATAN,KAB. KOTABARU,Kalimantan Selatan,Juni,347.4,✅ SINKRON: Terpeta di dalam tren historis DIBI
150,BANJIR,2022,KALIMANTAN SELATAN,KAB. KOTABARU,Kalimantan Selatan,Juli,260.2,✅ SINKRON: Terpeta di dalam tren historis DIBI
151,BANJIR,2022,KALIMANTAN SELATAN,KAB. KOTABARU,Kalimantan Selatan,Agustus,250.1,✅ SINKRON: Terpeta di dalam tren historis DIBI
152,BANJIR,2022,KALIMANTAN SELATAN,KAB. KOTABARU,Kalimantan Selatan,September,339.4,✅ SINKRON: Terpeta di dalam tren historis DIBI
153,BANJIR,2022,KALIMANTAN SELATAN,KAB. KOTABARU,Kalimantan Selatan,Oktober,488.1,✅ SINKRON: Terpeta di dalam tren historis DIBI



💾 File bersih siap pakai disimpan dengan nama: 'DATABASE_FINAL_TRI_SILO_READY.csv'


In [31]:
# ==============================================================================
# SEL: SANITY CHECK & FILTER RENTANG WAKTU (2010 - 2026)
# ==============================================================================
import pandas as pd

# Load data hasil penyaringan provinsi sebelumnya
df_final = pd.read_csv("DATABASE_FINAL_TRI_SILO_READY.csv")

# 1. Cetak rentang tahun yang saat ini ada di dalam data
tahun_min = df_final['tahun_kejadian'].min()
tahun_max = df_final['tahun_kejadian'].max()

print("🔍 Jalur Audit Rentang Waktu Pipeline:")
print(f"   • Tahun paling lampau yang terdeteksi: {tahun_min}")
print(f"   • Tahun paling baru yang terdeteksi : {tahun_max}")
print("-" * 60)

# 2. EKSEKUSI FILTER: Potong paksa agar HANYA meloloskan tahun >= 2010
df_2010_up = df_final[(df_final['tahun_kejadian'] >= 2010) & (df_final['tahun_kejadian'] <= 2026)].copy()

print(f"📊 Jumlah baris sebelum filter 2010: {len(df_final)} baris.")
print(f"🎯 Jumlah baris SETELAH dikunci (2010-2026): {len(df_2010_up)} baris.")

# 3. Tampilkan rangguman distribusi data per tahun untuk memastikan
print("\n📈 Distribusi Dokumen Terintegrasi Per Tahun (Post-Filter):")
print(df_2010_up['tahun_kejadian'].value_counts().sort_index())

# 4. Overwrite file master agar benar-benar steril dari data jadul
df_2010_up.to_csv("DATABASE_FINAL_TRI_SILO_READY.csv", index=False)
print("\n🔒 AMAN! Database telah disterilisasi. Rentang waktu resmi dikunci pada 2010-2026.")

🔍 Jalur Audit Rentang Waktu Pipeline:
   • Tahun paling lampau yang terdeteksi: 2017
   • Tahun paling baru yang terdeteksi : 2024
------------------------------------------------------------
📊 Jumlah baris sebelum filter 2010: 8688 baris.
🎯 Jumlah baris SETELAH dikunci (2010-2026): 8688 baris.

📈 Distribusi Dokumen Terintegrasi Per Tahun (Post-Filter):
tahun_kejadian
2017    1824
2021     456
2022      12
2023    1824
2024    4572
Name: count, dtype: int64

🔒 AMAN! Database telah disterilisasi. Rentang waktu resmi dikunci pada 2010-2026.
